In [ ]:
from torch import nn
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
# data = pd.read_csv('Electric_Production.csv')
# data = data.dropna()
# data = data.drop(columns='DATE').to_numpy()
# data = np.concatenate(data)


In [ ]:
# Read and preprocess data
data = pd.read_csv('Electric_Production.csv')
data = data.dropna()
data = data.drop(columns='DATE').to_numpy()
data = np.concatenate(data)

# Create moving average
window_size = 20
row = np.convolve(data, np.ones(window_size) / window_size, mode='valid')

# Create sliding windows
bigArray = np.lib.stride_tricks.sliding_window_view(row, 21)

# Prepare input and output arrays
outputX = []
outputY = []
for group in bigArray:
    x = group[:20].reshape(-1,1)
    y = group[-1]
    outputX.append(x)
    outputY.append(y)
outputX = np.array(outputX)
outputY = np.array(outputY).reshape(-1,1)  # Ensure outputY is 2D

# Split data into training and test sets
train_size = int(len(outputX) * 0.8)
X_train, X_test = outputX[:train_size], outputX[train_size:]
y_train, y_test = outputY[:train_size], outputY[train_size:]

# Fit scaler on training data only
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(X_train.reshape(-1, 1))

# Transform training and test data
X_train_scaled = scaler.transform(X_train.reshape(-1,1)).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1,1)).reshape(X_test.shape)
y_train_scaled = scaler.transform(y_train)
y_test_scaled = scaler.transform(y_test)

# Convert data to tensors
X_train_torch = torch.Tensor(X_train_scaled)
X_test_torch = torch.Tensor(X_test_scaled)
y_train_torch = torch.Tensor(y_train_scaled)
y_test_torch = torch.Tensor(y_test_scaled)

# Create DataLoader
train_dataset = TensorDataset(X_train_torch, y_train_torch)
trainLoader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)


In [ ]:
# import math
# window_size = 20
# row = np.convolve(data[~np.isnan(data)], np.ones(window_size) / window_size, mode='valid')
# bigArray = np.lib.stride_tricks.sliding_window_view(row, 21)
# outputX = []
# outputY = []
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler(feature_range=(-1,1))
# for group in bigArray:
#     x = group[:20].reshape(-1,1)
#     y = group[-1].reshape(-1,1)
#     x = scaler.fit_transform(x)
#     y = scaler.transform(y)
#     if np.any(np.isnan(x)) or np.any(np.isnan(y)):
#         continue

#     outputX.append(x)
#     outputY.append(y)
# outputX, outputY = np.array(outputX), np.array(outputY)
# X_train, X_test, y_train, y_test = outputX[:math.floor(.8*len(outputX))], outputX[math.floor(.8*len(outputX)):],outputY[:math.floor(.8*len(outputY))], outputY[math.floor(.8*len(outputY)):]
# X_train, X_test, y_train, y_test = torch.Tensor(X_train), torch.Tensor(X_test), torch.Tensor(y_train), torch.Tensor(y_test)
# trainLoader = DataLoader(dataset=TensorDataset(X_train, y_train), batch_size=32)

In [ ]:
class ZachPerceptron(nn.Module):
    def __init__(self, in_features):
        super(ZachPerceptron, self).__init__()
        self.weights = nn.Parameter(data=torch.rand(in_features, 1))
        self.bias = nn.Parameter(data=torch.rand(1,1))
    def forward(self,x):
        return x @ self.weights + self.bias
class ZachLinear(nn.Module):
    def __init__(self,in_features, out_features):
        super(ZachLinear, self).__init__()
        self.weights = nn.Parameter(data=torch.rand(in_features, out_features))
        self.bias = nn.Parameter(data=torch.rand(out_features))
    def forward(self,x):
        x = torch.matmul(x,self.weights)
        wbias = torch.add(x,self.bias)
        return wbias
class ZachRNN(nn.Module):
    def __init__(self,input_size, hidden_size):
        super(ZachRNN, self).__init__()
        self.hidden_state = hidden_size
        self.inputWeight = nn.Parameter(torch.randn(input_size, hidden_size))
        self.hiddenWeight = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.bias = nn.Parameter(torch.randn(hidden_size))
        self.outputWeight = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.biasOutput = nn.Parameter(torch.randn(hidden_size))
        self.activation = nn.Tanh()
        nn.init.xavier_uniform_(self.inputWeight)
        nn.init.xavier_uniform_(self.hiddenWeight)
        nn.init.xavier_uniform_(self.outputWeight)
    def forward(self,x):
        batch, sequenceLength, _ = x.shape
        hidden = torch.zeros(batch, self.hidden_state).to(x.device)
        outputs = []
        for i in range(sequenceLength):
            input_current = x[:, i, :]  
            combined = torch.add(
                torch.matmul(input_current, self.inputWeight) + torch.matmul(hidden, self.hiddenWeight),
                self.bias
            ) 
            hidden = self.activation(combined)
            output = torch.add(torch.matmul(hidden, self.outputWeight), self.biasOutput)
            outputs.append(output)
        return torch.stack(outputs, dim=1), hidden
class ZachLSTM(nn.Module):
    def __init__(self,input_size, hidden_size):
        super(ZachLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.inputWeights = nn.Parameter(data=torch.randn(input_size, hidden_size*4))
        self.shortTermHiddenWeights = nn.Parameter(data=torch.randn(hidden_size, hidden_size*4))
        self.bias = nn.Parameter(data=torch.randn(hidden_size*4))
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()
        nn.init.xavier_uniform_(self.inputWeights)
        nn.init.xavier_uniform_(self.shortTermHiddenWeights)
        nn.init.zeros_(self.bias)
    def forward(self,x):
        batch_size, sequenceLength, _ = x.shape
        shortHidden = torch.zeros(batch_size, self.hidden_size)
        longHidden = torch.zeros(batch_size,self.hidden_size)
        outputs = []
        for i in range(sequenceLength):
            inputCurrent = x[:, i, :]
            inputsPostWeight = inputCurrent @ self.inputWeights
            shortHiddenPostWeight = shortHidden @ self.shortTermHiddenWeights
            combined = inputsPostWeight + shortHiddenPostWeight + self.bias
            forgetGateInput, percentageInputInput, LogitInputInput, percentageOutputInput = torch.split(combined, self.hidden_size, dim=1)
            forgetGateInput = self.sigmoid(forgetGateInput)
            percentageInputInput = self.sigmoid(percentageInputInput)
            LogitInputInput = self.tanh(LogitInputInput)
            percentageOutputInput = self.sigmoid(percentageOutputInput)
            longHidden = longHidden * forgetGateInput
            longHidden = longHidden + (percentageInputInput * LogitInputInput)
            shortHidden = self.tanh(longHidden) * percentageOutputInput
            outputs.append(shortHidden)
        outputs = torch.stack(outputs, dim=1)
        return outputs, (shortHidden, longHidden)

class ZachGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(ZachGRU, self).__init__()
        self.hidden_size = hidden_size
        self.inputWeights = nn.Parameter(data=torch.randn(input_size, hidden_size*3))
        self.hiddenWeights = nn.Parameter(data=torch.randn(hidden_size,hidden_size*3))
        self.bias = nn.Parameter(data=torch.randn(hidden_size*3))
        nn.init.xavier_uniform_(self.inputWeights)
        nn.init.xavier_uniform_(self.hiddenWeights)
        nn.init.zeros_(self.bias)
    def forward(self,x):
        batchLength, sequenceLength, _ = x.shape
        hidden = torch.zeros(batchLength,self.hidden_size)
        outputs = []
        for i in range(sequenceLength):
            inputCurrent = x[:,i,:]
            combined = inputCurrent @ self.inputWeights + hidden @ self.hiddenWeights + self.bias
            updateGate, resetGate, candidate_pre = torch.split(combined, self.hidden_size, dim=1)
            resetGate = torch.sigmoid(resetGate)
            updateGate = torch.sigmoid(updateGate)
            hidden = hidden * updateGate
            hidden = hidden + torch.tanh(resetGate*hidden@ self.hiddenWeights[:, 2 * self.hidden_size:]+candidate_pre)*(1-updateGate)
            outputs.append(hidden)
        outputs = torch.stack(outputs, dim=1)
        return outputs, hidden




class ElectricityProduction(nn.Module):
    def __init__(self):
        super(ElectricityProduction, self).__init__()
        self.rnn = ZachGRU(input_size=1, hidden_size=64)
        self.fc = ZachLinear(64,1)
    def forward(self,x):
        x,_ = self.rnn(x)
        x = self.fc(x[:,-1])
        return x

In [ ]:
model = ElectricityProduction()
epochs = 1000
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-3)
for epoch in tqdm(range(epochs)):
    model.train()
    total_loss = 0
    for X_batch, y_batch in trainLoader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(trainLoader):.6f}')


In [ ]:
model.eval()
with torch.inference_mode():
    y_pred_scaled = model(X_test_torch)
    y_pred = scaler.inverse_transform(y_pred_scaled.numpy())
    y_test_actual = scaler.inverse_transform(y_test_scaled)

plt.figure(figsize=(12, 6))
plt.plot(y_test_actual, label='Actual Data', color='g')
plt.plot(y_pred, label='Predicted Data', color='b')
plt.legend()
plt.show()

In [ ]:
X_test.shape

In [ ]:
X_test.flatten(start_dim=0, end_dim=1)

In [ ]:
X_test

In [ ]:
import math
window_size = 20
row = np.convolve(data[~np.isnan(data)], np.ones(window_size) / window_size, mode='valid')
bigArray = np.lib.stride_tricks.sliding_window_view(row, 21)
outputX = []
outputY = []
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1,1))
for group in bigArray:
    x = group[:20].reshape(-1,1)
    y = group[-1].reshape(-1,1)
    # x = scaler.fit_transform(x)
    # y = scaler.transform(y)
    if np.any(np.isnan(x)) or np.any(np.isnan(y)):
        continue

    outputX.append(x)
    outputY.append(y)
outputX, outputY = np.array(outputX), np.array(outputY)
X_train, X_test, y_train, y_test = outputX[:math.floor(.8*len(outputX))], outputX[math.floor(.8*len(outputX)):],outputY[:math.floor(.8*len(outputY))], outputY[math.floor(.8*len(outputY)):]
X_train, X_test, y_train, y_test = torch.Tensor(X_train), torch.Tensor(X_test), torch.Tensor(y_train), torch.Tensor(y_test)
trainLoader = DataLoader(dataset=TensorDataset(X_train, y_train), batch_size=32)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.arange(72), data[-72:], color='g')
plt.plot(np.arange(72), torch.Tensor(preds).squeeze())